# Fake Review Detection - Step 6: DeBERTa-v3-base Text-Only Training
**Project**: `fake-review-detection-v2`  
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

This notebook fine-tunes Microsoft DeBERTa-v3-base (`microsoft/deberta-v3-base`) strictly on raw review text from the reviewer-held-out training split, evaluates out-of-fold generalization on the validation split, and tracks validation metrics across training epochs.

### Key Guidelines & Constraints:
- **Predictive Feature**: `review_text` only (no reviewer IDs, product IDs, or tabular metadata).
- **Target Mapping**: `genuine = 0`, `fake = 1`.
- **Test Set Protection**: `data/splits/test.csv` is strictly reserved and NEVER loaded, evaluated, or tuned against.
- **Hardware**: Optimized for Google Colab with NVIDIA T4 GPU (or V100 / A100).

## 1. Environment & Hardware Verification
Detects runtime environment, checks for GPU acceleration, CUDA device properties, and memory availability.

In [ ]:
import os
import sys
import platform
import torch

print("=" * 65)
print("SYSTEM & HARDWARE ENVIRONMENT")
print("=" * 65)
print(f"Python Version   : {sys.version.split()[0]} ({platform.python_implementation()})")
print(f"Platform         : {platform.platform()}")

IN_COLAB = "google.colab" in sys.modules
print(f"Google Colab     : {'Yes' if IN_COLAB else 'No (Local/Standalone)'}")

gpu_available = torch.cuda.is_available()
print(f"CUDA Available   : {gpu_available}")
if gpu_available:
    device_name = torch.cuda.get_device_name(0)
    device_cap = torch.cuda.get_device_capability(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU Device       : {device_name} (Compute {device_cap[0]}.{device_cap[1]})")
    print(f"Total VRAM       : {vram_gb:.2f} GB")
else:
    print("\n[!] WARNING: No CUDA GPU detected!")
    print("Fine-tuning DeBERTa on CPU is computationally prohibitive.")
    print("In Google Colab: Go to Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU.")
print("=" * 65)

## 2. Dependencies Installation
Installs `transformers`, `datasets`, `accelerate`, `evaluate`, and requirements for DeBERTa tokenization and evaluation.

In [ ]:
# Install required libraries for Colab GPU execution
!pip install --quiet \
    "transformers>=4.35.0" \
    "datasets>=2.14.0" \
    "accelerate>=0.26.0" \
    "evaluate>=0.4.0" \
    "scikit-learn>=1.3.0" \
    "pandas>=2.0.0" \
    "numpy>=1.24.0" \
    "pyyaml>=6.0"

import transformers
import datasets
import accelerate
import sklearn

print("[OK] Installed Package Versions:")
print(f"  - Transformers : {transformers.__version__}")
print(f"  - Datasets     : {datasets.__version__}")
print(f"  - Accelerate   : {accelerate.__version__}")
print(f"  - Scikit-learn : {sklearn.__version__}")
print(f"  - PyTorch      : {torch.__version__}")

## 3. Configuration & Google Drive Integration
Configure project root directory, paths to reviewer-held-out splits, and initial hyperparameters.

In [ ]:
import os
import sys

# Mount Google Drive if in Colab
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    # Default path: adjust if your project is in a different Drive folder
    PROJECT_ROOT = "/content/drive/MyDrive/fake-review-detection-v2"
    if not os.path.exists(PROJECT_ROOT):
        # Fallback to local runtime current directory
        PROJECT_ROOT = "."
else:
    PROJECT_ROOT = "."

print(f"Configured PROJECT_ROOT: {os.path.abspath(PROJECT_ROOT)}")

# Split Paths
TRAIN_PATH = os.path.join(PROJECT_ROOT, "data", "splits", "train.csv")
VAL_PATH = os.path.join(PROJECT_ROOT, "data", "splits", "validation.csv")

# Test set protection: Define path only for split existence check; NEVER load during training/tuning
TEST_PATH_RESERVED = os.path.join(PROJECT_ROOT, "data", "splits", "test.csv")

# Artifact & Checkpoint Paths
ARTIFACTS_DIR = os.path.join(PROJECT_ROOT, "training", "artifacts", "deberta_text")
CHECKPOINTS_DIR = os.path.join(PROJECT_ROOT, "training", "artifacts", "deberta_text", "checkpoints")
REPORTS_DIR = os.path.join(PROJECT_ROOT, "reports")

os.makedirs(ARTIFACTS_DIR, exist_ok=True)
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

# Training Hyperparameters (Configurable)
CONFIG = {
    "model_name": "microsoft/deberta-v3-base",
    "seed": 42,
    "max_length": 256,
    "per_device_train_batch_size": 8,
    "per_device_eval_batch_size": 16,
    "gradient_accumulation_steps": 2,  # Effective batch size: 16
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    "num_train_epochs": 3,
    "warmup_ratio": 0.10,
    "fp16": torch.cuda.is_available(),
    "early_stopping_patience": 2,
    "metric_for_best_model": "f1",
}

print("\nModel & Initial Training Configuration:")
for k, v in CONFIG.items():
    print(f"  - {k:<30}: {v}")

## 4. Dataset Loading, Integrity Verification & Warmup Calculation
Load `train.csv` and `validation.csv`. Validate target mapping (`genuine = 0`, `fake = 1`) and dynamically compute exact `warmup_steps` from dataset size, batch size, gradient accumulation, device count, and epochs.
*Test split is verified on disk but strictly NOT loaded to guarantee zero leakage.*

In [ ]:
import pandas as pd
import numpy as np
import math

# Verify files exist
assert os.path.exists(TRAIN_PATH), f"Train split missing: {TRAIN_PATH}"
assert os.path.exists(VAL_PATH), f"Val split missing: {VAL_PATH}"
assert os.path.exists(TEST_PATH_RESERVED), f"Test split missing: {TEST_PATH_RESERVED}"

# Load training and validation sets only
df_train = pd.read_csv(TRAIN_PATH)
df_val = pd.read_csv(VAL_PATH)

print("=" * 65)
print("SPLIT DATASET SUMMARY")
print("=" * 65)
print(f"Train samples : {len(df_train):,} reviews")
print(f"Val samples   : {len(df_val):,} reviews")
print(f"Test split    : RESERVED (data/splits/test.csv verified on disk, NOT loaded)")

# Verify no missing text or labels
assert df_train["review_text"].notna().all(), "Missing text in train split!"
assert df_val["review_text"].notna().all(), "Missing text in val split!"
assert df_train["label"].isin(["genuine", "fake"]).all(), "Invalid labels in train!"
assert df_val["label"].isin(["genuine", "fake"]).all(), "Invalid labels in val!"

# Standardize labels to binary integers: genuine = 0, fake = 1
label_map = {"genuine": 0, "fake": 1}
df_train["target"] = df_train["label"].map(label_map)
df_val["target"] = df_val["label"].map(label_map)

print("\nClass Distributions:")
print(f"  - Train Genuine (0) : {(df_train['target'] == 0).sum():,} ({(df_train['target'] == 0).mean()*100:.2f}%)")
print(f"  - Train Fake (1)    : {(df_train['target'] == 1).sum():,} ({(df_train['target'] == 1).mean()*100:.2f}%)")
print(f"  - Val Genuine (0)   : {(df_val['target'] == 0).sum():,} ({(df_val['target'] == 0).mean()*100:.2f}%)")
print(f"  - Val Fake (1)      : {(df_val['target'] == 1).sum():,} ({(df_val['target'] == 1).mean()*100:.2f}%)")

# Calculate warmup_steps dynamically for universal TrainingArguments compatibility
num_train_samples = len(df_train)
batch_size = CONFIG["per_device_train_batch_size"]
grad_accum = CONFIG["gradient_accumulation_steps"]
device_count = max(1, torch.cuda.device_count()) if torch.cuda.is_available() else 1
num_epochs = CONFIG["num_train_epochs"]
warmup_ratio = CONFIG.get("warmup_ratio", 0.10)

effective_batch_size = batch_size * grad_accum * device_count
steps_per_epoch = math.ceil(num_train_samples / effective_batch_size)
total_training_steps = steps_per_epoch * num_epochs
calculated_warmup_steps = int(round(total_training_steps * warmup_ratio))
CONFIG["warmup_steps"] = calculated_warmup_steps

print("\nWarmup Steps Calculation:")
print(f"  - Training samples        : {num_train_samples:,}")
print(f"  - Per-device batch size   : {batch_size}")
print(f"  - Gradient accumulation   : {grad_accum}")
print(f"  - Device count            : {device_count}")
print(f"  - Effective batch size    : {effective_batch_size}")
print(f"  - Steps per epoch         : {steps_per_epoch:,}")
print(f"  - Total training steps    : {total_training_steps:,}")
print(f"  - Warmup ratio            : {warmup_ratio:.2f}")
print(f"  - Configured warmup_steps : {calculated_warmup_steps:,}")
print("=" * 65)

## 5. Tokenization & Dynamic Padding
Initialize the DeBERTa-v3 tokenizer and convert pandas DataFrames into Hugging Face `Dataset` objects with dynamic batch padding.
*Only `review_text` and `target` (mapped to `label`) are used. Metadata, reviewer IDs, and product IDs are omitted.*

In [ ]:
from transformers import AutoTokenizer, DataCollatorWithPadding
from datasets import Dataset

print(f"Loading tokenizer: {CONFIG['model_name']}...")
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"], use_fast=True)

def tokenize_function(examples):
    return tokenizer(
        examples["review_text"],
        truncation=True,
        max_length=CONFIG["max_length"],
    )

# Prepare Hugging Face datasets with review_text as ONLY predictive input
train_dataset = Dataset.from_pandas(df_train[["review_text", "target"]].rename(columns={"target": "label"}))
val_dataset = Dataset.from_pandas(df_val[["review_text", "target"]].rename(columns={"target": "label"}))

print("Tokenizing train dataset...")
train_tokenized = train_dataset.map(tokenize_function, batched=True, remove_columns=["review_text"])
print("Tokenizing validation dataset...")
val_tokenized = val_dataset.map(tokenize_function, batched=True, remove_columns=["review_text"])

# Dynamic batch padding collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("\n[OK] Tokenization complete.")
print(f"  - Train dataset features : {list(train_tokenized.features.keys())}")
print(f"  - Sample input_ids length: {len(train_tokenized[0]['input_ids'])} tokens")

## 6. Evaluation Metrics Definition
Define comprehensive evaluation metrics computed at the end of each epoch:
- F1-Score (Binary, positive class = Fake)
- Precision & Recall
- PR-AUC (Average Precision)
- ROC-AUC

In [ ]:
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score
)
from scipy.special import softmax

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = softmax(logits, axis=-1)[:, 1]
    preds = np.argmax(logits, axis=-1)

    precision = precision_score(labels, preds, zero_division=0)
    recall = recall_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)
    roc_auc = roc_auc_score(labels, probs)
    pr_auc = average_precision_score(labels, probs)

    return {
        "precision": float(round(precision, 4)),
        "recall": float(round(recall, 4)),
        "f1": float(round(f1, 4)),
        "roc_auc": float(round(roc_auc, 4)),
        "pr_auc": float(round(pr_auc, 4)),
    }

print("[OK] Metric computation function ready.")

## 7. Diagnostic Check: TrainingArguments Compatibility
Inspects supported `TrainingArguments` parameters in the installed Transformers library and validates that `TrainingArguments` can be instantiated without errors.

In [ ]:
import inspect
from transformers import TrainingArguments

sig = inspect.signature(TrainingArguments.__init__)
eval_param = "eval_strategy" if "eval_strategy" in sig.parameters else "evaluation_strategy"
diagnostic_eval_kwargs = {eval_param: "epoch"}

print("=" * 65)
print("TRAININGARGUMENTS COMPATIBILITY DIAGNOSTIC")
print("=" * 65)
print(f"Transformers Version : {transformers.__version__}")
print(f"Warmup Support       : warmup_steps ({'supported' if 'warmup_steps' in sig.parameters else 'unsupported'})")
print(f"warmup_ratio in args : {'warmup_ratio' in sig.parameters}")
print(f"Evaluation Parameter : {eval_param} = 'epoch'")

# Test-instantiate TrainingArguments
try:
    test_args = TrainingArguments(
        output_dir=CHECKPOINTS_DIR,
        save_strategy="epoch",
        learning_rate=CONFIG["learning_rate"],
        per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
        per_device_eval_batch_size=CONFIG["per_device_eval_batch_size"],
        gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
        num_train_epochs=CONFIG["num_train_epochs"],
        weight_decay=CONFIG["weight_decay"],
        warmup_steps=CONFIG["warmup_steps"],
        fp16=CONFIG["fp16"],
        load_best_model_at_end=True,
        metric_for_best_model=CONFIG["metric_for_best_model"],
        greater_is_better=True,
        save_total_limit=2,
        logging_steps=100,
        seed=CONFIG["seed"],
        report_to="none",
        **diagnostic_eval_kwargs
    )
    print("\n[OK] Diagnostic Passed: TrainingArguments successfully instantiated!")
    print(f"  - warmup_steps: {test_args.warmup_steps}")
    print(f"  - learning_rate: {test_args.learning_rate}")
    print(f"  - eval_strategy: {getattr(test_args, 'eval_strategy', getattr(test_args, 'evaluation_strategy', None))}")
except Exception as diag_err:
    print(f"\n[FAIL] Diagnostic Failed: {diag_err}")
    raise diag_err
print("=" * 65)

## 8. Model Initialization & Trainer Setup
Initialize `AutoModelForSequenceClassification` with 2 output labels (`0`: Genuine, `1`: Fake), configure `TrainingArguments` with checkpoint saving and early stopping.

In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed
)

set_seed(CONFIG["seed"])

print(f"Loading pretrained model: {CONFIG['model_name']}...")
model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG["model_name"],
    num_labels=2,
    id2label={0: "genuine", 1: "fake"},
    label2id={"genuine": 0, "fake": 1},
)

# Cross-version compatibility for evaluation strategy argument
eval_kwargs = {}
sig = inspect.signature(TrainingArguments.__init__)
if "eval_strategy" in sig.parameters:
    eval_kwargs["eval_strategy"] = "epoch"
elif "evaluation_strategy" in sig.parameters:
    eval_kwargs["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(
    output_dir=CHECKPOINTS_DIR,
    save_strategy="epoch",
    learning_rate=CONFIG["learning_rate"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=CONFIG["per_device_eval_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    num_train_epochs=CONFIG["num_train_epochs"],
    weight_decay=CONFIG["weight_decay"],
    warmup_steps=CONFIG["warmup_steps"],
    fp16=CONFIG["fp16"],
    load_best_model_at_end=True,
    metric_for_best_model=CONFIG["metric_for_best_model"],
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=100,
    seed=CONFIG["seed"],
    report_to="none",
    **eval_kwargs
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=CONFIG["early_stopping_patience"])
    ],
)

print("[OK] Trainer initialized and ready for execution.")

## 9. Fine-Tuning Execution
Execute fine-tuning on the reviewer-held-out training split. Checkpoints will be saved at the end of each epoch to Google Drive.

In [ ]:
print("=" * 65)
print("STARTING DEBERTA-V3 FINE-TUNING")
print("=" * 65)

try:
    train_result = trainer.train()
    print("\n[OK] Training finished successfully!")
    print(f"Total Training Time: {train_result.metrics.get('train_runtime', 0):.2f} seconds")
except torch.cuda.OutOfMemoryError as oom:
    print("\n[ERROR] CUDA Out-of-Memory Error!")
    print("Suggestions to resolve:")
    print("  1. Reduce per_device_train_batch_size to 4 and increase gradient_accumulation_steps to 4.")
    print("  2. Reduce max_length to 192 or 128.")
    raise oom
except Exception as e:
    print(f"\n[ERROR] Error during training: {e}")
    raise e

## 10. Comprehensive Validation Evaluation & Threshold Tuning
Evaluate the fine-tuned checkpoint on the validation set.
1. Evaluate at default threshold `0.50`.
2. Tune decision threshold strictly on validation set to find the optimal F1 threshold.
*Test set is NEVER touched.*

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import json

print("Running validation inference with best checkpoint...")
raw_preds = trainer.predict(val_tokenized)
val_logits = raw_preds.predictions
val_labels = raw_preds.label_ids
val_probs = softmax(val_logits, axis=-1)[:, 1]

# 1. Default Threshold (0.50)
y_pred_05 = (val_probs >= 0.50).astype(int)
roc_auc_val = float(roc_auc_score(val_labels, val_probs))
pr_auc_val = float(average_precision_score(val_labels, val_probs))
p_05 = float(precision_score(val_labels, y_pred_05, zero_division=0))
r_05 = float(recall_score(val_labels, y_pred_05, zero_division=0))
f1_05 = float(f1_score(val_labels, y_pred_05, zero_division=0))
cm_05 = confusion_matrix(val_labels, y_pred_05).tolist()

# 2. Optimal Threshold Search on Validation Data
best_thresh = 0.50
best_f1 = f1_05
for t in np.arange(0.10, 0.91, 0.01):
    t_round = round(float(t), 2)
    preds_t = (val_probs >= t_round).astype(int)
    f1_t = float(f1_score(val_labels, preds_t, zero_division=0))
    if f1_t > best_f1:
        best_f1 = f1_t
        best_thresh = t_round

y_pred_best = (val_probs >= best_thresh).astype(int)
p_best = float(precision_score(val_labels, y_pred_best, zero_division=0))
r_best = float(recall_score(val_labels, y_pred_best, zero_division=0))
cm_best = confusion_matrix(val_labels, y_pred_best).tolist()

validation_summary = {
    "model_name": CONFIG["model_name"],
    "roc_auc": round(roc_auc_val, 4),
    "pr_auc": round(pr_auc_val, 4),
    "threshold_0_5": {
        "threshold": 0.50,
        "precision": round(p_05, 4),
        "recall": round(r_05, 4),
        "f1_score": round(f1_05, 4),
        "confusion_matrix": {"tn": cm_05[0][0], "fp": cm_05[0][1], "fn": cm_05[1][0], "tp": cm_05[1][1]},
    },
    "optimal_threshold": {
        "threshold": best_thresh,
        "precision": round(p_best, 4),
        "recall": round(r_best, 4),
        "f1_score": round(best_f1, 4),
        "confusion_matrix": {"tn": cm_best[0][0], "fp": cm_best[0][1], "fn": cm_best[1][0], "tp": cm_best[1][1]},
    }
}

print("\n" + "=" * 65)
print("DEBERTA-V3 VALIDATION PERFORMANCE RESULTS")
print("=" * 65)
print(f"ROC-AUC           : {roc_auc_val:.4f}")
print(f"PR-AUC            : {pr_auc_val:.4f}")
print(f"[Threshold 0.50]  : Precision={p_05:.4f}, Recall={r_05:.4f}, F1={f1_05:.4f}")
print(f"Confusion Matrix  : TN={cm_05[0][0]}, FP={cm_05[0][1]}, FN={cm_05[1][0]}, TP={cm_05[1][1]}")
print(f"\n[Optimal {best_thresh:.2f}] : Precision={p_best:.4f}, Recall={r_best:.4f}, F1={best_f1:.4f}")
print(f"Confusion Matrix  : TN={cm_best[0][0]}, FP={cm_best[0][1]}, FN={cm_best[1][0]}, TP={cm_best[1][1]}")
print("=" * 65)

## 11. Save Artifacts & Generate Validation Report
Save the final model weights, tokenizer files, configuration, and validation summary to `training/artifacts/deberta_text/` and `reports/deberta_validation_report.txt`.

In [ ]:
# Save best model and tokenizer
print(f"Saving best model to: {ARTIFACTS_DIR}...")
trainer.save_model(ARTIFACTS_DIR)
tokenizer.save_pretrained(ARTIFACTS_DIR)

# Save training configuration
config_out_path = os.path.join(ARTIFACTS_DIR, "training_config.json")
with open(config_out_path, "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2)
print(f"Saved training config JSON: {config_out_path}")

# Save metrics JSON
metrics_out_path = os.path.join(ARTIFACTS_DIR, "validation_metrics.json")
with open(metrics_out_path, "w", encoding="utf-8") as f:
    json.dump(validation_summary, f, indent=2)
print(f"Saved validation metrics JSON: {metrics_out_path}")

# Load baseline validation metrics for comparison
baseline_metrics_path = os.path.join(PROJECT_ROOT, "training", "artifacts", "baseline", "validation_metrics.json")
baseline_data = {}
if os.path.exists(baseline_metrics_path):
    with open(baseline_metrics_path, "r", encoding="utf-8") as f:
        baseline_data = json.load(f)

# Write comparison report
report_out_path = os.path.join(REPORTS_DIR, "deberta_validation_report.txt")
b_05 = baseline_data.get("threshold_0_5", {})

report_lines = [
    "=" * 78,
    " FAKE REVIEW DETECTION - DEBERTA-V3 VALIDATION & COMPARISON REPORT",
    "=" * 78,
    f"Model Architecture  : {CONFIG['model_name']} (Fine-Tuned)",
    f"Predictive Input    : review_text (Text-Only Transformer)",
    f"Target Mapping      : genuine = 0, fake = 1",
    f"Training Set        : data/splits/train.csv ({len(df_train):,} reviews)",
    f"Validation Set      : data/splits/validation.csv ({len(df_val):,} reviews)",
    f"Test Set Status     : RESERVED (data/splits/test.csv was NOT evaluated)",
    "-" * 78,
    "1. VALIDATION BENCHMARK COMPARISON",
    "-" * 78,
    f"{'Metric':<25} | {'Baseline (TF-IDF+LGBM)':<24} | {'DeBERTa-v3-base':<20}",
    "-" * 78,
    f"{'ROC-AUC':<25} | {str(baseline_data.get('roc_auc', 'N/A')):<24} | {validation_summary['roc_auc']:<20.4f}",
    f"{'PR-AUC':<25} | {str(baseline_data.get('pr_auc', 'N/A')):<24} | {validation_summary['pr_auc']:<20.4f}",
    f"{'Precision (T=0.50)':<25} | {str(b_05.get('precision', 'N/A')):<24} | {validation_summary['threshold_0_5']['precision']:<20.4f}",
    f"{'Recall (T=0.50)':<25} | {str(b_05.get('recall', 'N/A')):<24} | {validation_summary['threshold_0_5']['recall']:<20.4f}",
    f"{'F1-Score (T=0.50)':<25} | {str(b_05.get('f1_score', 'N/A')):<24} | {validation_summary['threshold_0_5']['f1_score']:<20.4f}",
    f"{'Optimal F1':<25} | {str(baseline_data.get('optimal_threshold', {}).get('f1_score', 'N/A')):<24} | {validation_summary['optimal_threshold']['f1_score']:<20.4f}",
    "-" * 78,
    "2. ANALYSIS & OBSERVATIONS",
    "-" * 78,
    "- Contextual Representation:",
    "  DeBERTa-v3 utilizes disentangled attention to capture contextual word dependencies",
    "  beyond bag-of-words n-gram frequencies.",
    "- Handling of Template Clusters:",
    "  While DeBERTa improves semantic sensitivity, mixed-label template clusters",
    "  (nearly identical text with opposing ground-truth labels) still represent a fundamental",
    "  ceiling for any text-only model.",
    "- Step 7 Roadmap:",
    "  Multimodal fusion combining DeBERTa-v3 embeddings with tabular metadata",
    "  (verified_purchase, historical review rates) will provide the missing orthogonal signals.",
    "=" * 78,
]

with open(report_out_path, "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines) + "\n")
print(f"[OK] Saved validation report: {report_out_path}")